
# Strong Long-Pretrained HAT Transfer to the Local 6-Band Dataset

## الهدف

نقل أفضل EMA Checkpoint من التدريب الطويل على WorldView-3 إلى بياناتك المحلية:

```text
WV3 source:
8 MS bands + PAN

Local target:
6 MS bands + PAN
```

ثم Fine-tuning أقوى من التجربة السابقة باستخدام:

```text
Stage A: Adapt input/output/refinement layers
Stage B: Full HAT fine-tuning
Charbonnier + SSIM + spectral-angle + LR consistency
EMA
Long training
Automatic resume
Best PSNR and best balanced checkpoints
```

## المقارنة النهائية

```text
Bicubic
Current WV3-Transfer HAT
New Long-Pretrained HAT Transfer
```


In [ ]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("فعّل T4 GPU.")

device = torch.device("cuda")
torch.set_float32_matmul_precision("high")

print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

from pathlib import Path
import shutil
import subprocess
import importlib.util

%cd /content

HAT_REPO = Path("/content/HAT")

if HAT_REPO.exists():
    shutil.rmtree(HAT_REPO)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/XPixelGroup/HAT.git",
        str(HAT_REPO),
    ],
    check=True,
)

!pip install -q einops timm scikit-image pytorch-msssim pandas matplotlib tqdm

original_arch = HAT_REPO / "hat" / "archs" / "hat_arch.py"
standalone_arch = Path("/content/hat_arch_standalone.py")

source = original_arch.read_text(encoding="utf-8")

source = source.replace(
    "from basicsr.utils.registry import ARCH_REGISTRY",
    """class _SimpleRegistry:
    def register(self):
        def decorator(obj):
            return obj
        return decorator
ARCH_REGISTRY = _SimpleRegistry()"""
)

source = source.replace(
    "from basicsr.archs.arch_util import to_2tuple, trunc_normal_",
    "from timm.layers import to_2tuple, trunc_normal_"
)

standalone_arch.write_text(source, encoding="utf-8")

spec = importlib.util.spec_from_file_location(
    "hat_arch_standalone",
    standalone_arch,
)

hat_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hat_module)

HAT = hat_module.HAT

print("Official HAT imported.")


In [ ]:

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Super_Resolution_28-07-2026"
)

PATCH_DIR = PROJECT_DIR / "Wald_Data_GSD" / "Patches"

TRAIN_DIR = PATCH_DIR / "train"
VAL_DIR = PATCH_DIR / "val"
TEST_DIR = PATCH_DIR / "test"

STATS_PATH = (
    PROJECT_DIR
    / "Fusion_Baseline_Results"
    / "train_normalization_stats.json"
)

LONG_WV3_CHECKPOINT = (
    PROJECT_DIR
    / "WV3_HAT_Long_Pretraining"
    / "best_psnr_wv3_hat_long.pth"
)

CURRENT_LOCAL_HAT_PATH = (
    PROJECT_DIR
    / "WV3_Transfer_HAT_Results"
    / "best_wv3_transfer_hat_6band.pth"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "WV3_Long_Transfer_HAT_6Band"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_PSNR_PATH = OUTPUT_DIR / "best_psnr_long_transfer_hat_6band.pth"
BEST_BALANCED_PATH = OUTPUT_DIR / "best_balanced_long_transfer_hat_6band.pth"
LAST_PATH = OUTPUT_DIR / "last_long_transfer_hat_6band.pth"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"
TEST_CSV_PATH = OUTPUT_DIR / "final_test_comparison.csv"
TEST_JSON_PATH = OUTPUT_DIR / "final_test_comparison.json"

for path in [
    TRAIN_DIR,
    VAL_DIR,
    TEST_DIR,
    STATS_PATH,
    LONG_WV3_CHECKPOINT,
    CURRENT_LOCAL_HAT_PATH,
]:
    print(path, "→", path.exists())

    if not path.exists():
        raise FileNotFoundError(path)


In [ ]:

import os
import json
import math
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

RUN_PROFILE = "smoke"
# "smoke", "long", "max"

SEED = 42
BATCH_SIZE = 1
ACCUMULATION_STEPS = 4
NUM_WORKERS = 0

STAGE_A_EPOCHS = 10

if RUN_PROFILE == "smoke":
    TOTAL_EPOCHS = 12
elif RUN_PROFILE == "long":
    TOTAL_EPOCHS = 80
elif RUN_PROFILE == "max":
    TOTAL_EPOCHS = 150
else:
    raise ValueError(RUN_PROFILE)

STAGE_A_LR = 1e-4
STAGE_B_LR = 2e-5
WEIGHT_DECAY = 1e-6
PATIENCE = 15

CHARBONNIER_EPS = 1e-3
SSIM_WEIGHT = 0.04
SAM_WEIGHT = 0.02
LR_CONSISTENCY_WEIGHT = 0.05

EMA_DECAY = 0.999
GRADIENT_CLIP = 1.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("Profile:", RUN_PROFILE)
print("Total epochs:", TOTAL_EPOCHS)
print("Stage A:", STAGE_A_EPOCHS)
print("Stage B:", max(TOTAL_EPOCHS - STAGE_A_EPOCHS, 0))


In [ ]:

with open(STATS_PATH, "r", encoding="utf-8") as file:
    stats = json.load(file)

MS_LOW = np.array(
    [item["p01"] for item in stats["ms"]],
    dtype=np.float32,
)[:, None, None]

MS_HIGH = np.array(
    [item["p99"] for item in stats["ms"]],
    dtype=np.float32,
)[:, None, None]

PAN_LOW = np.float32(stats["pan"]["p01"])
PAN_HIGH = np.float32(stats["pan"]["p99"])


def normalize_ms(array):
    return np.clip(
        (array.astype(np.float32) - MS_LOW)
        / np.maximum(MS_HIGH - MS_LOW, 1e-6),
        0,
        1,
    )


def normalize_pan(array):
    return np.clip(
        (array.astype(np.float32) - PAN_LOW)
        / max(float(PAN_HIGH - PAN_LOW), 1e-6),
        0,
        1,
    )


def tensor_to_dn(tensor):
    low = torch.from_numpy(MS_LOW).to(
        tensor.device,
        tensor.dtype,
    )

    high = torch.from_numpy(MS_HIGH).to(
        tensor.device,
        tensor.dtype,
    )

    return tensor * (high - low) + low


In [ ]:

from torch.utils.data import Dataset, DataLoader


class LocalWaldDataset(Dataset):
    def __init__(self, folder, augment=False):
        self.files = sorted(Path(folder).glob("*.npz"))
        self.augment = augment

        if not self.files:
            raise RuntimeError(f"No files in {folder}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        path = self.files[index]

        with np.load(path) as sample:
            lr_ms = torch.from_numpy(
                normalize_ms(sample["lr_ms"].copy())
            ).float()

            pan = torch.from_numpy(
                normalize_pan(sample["pan"].copy())
            ).float()

            target = torch.from_numpy(
                normalize_ms(sample["target_ms"].copy())
            ).float()

        if self.augment:
            if random.random() < 0.5:
                lr_ms = torch.flip(lr_ms, dims=[2])
                pan = torch.flip(pan, dims=[2])
                target = torch.flip(target, dims=[2])

            if random.random() < 0.5:
                lr_ms = torch.flip(lr_ms, dims=[1])
                pan = torch.flip(pan, dims=[1])
                target = torch.flip(target, dims=[1])

            rotations = random.randint(0, 3)

            if rotations:
                lr_ms = torch.rot90(lr_ms, rotations, dims=[1, 2])
                pan = torch.rot90(pan, rotations, dims=[1, 2])
                target = torch.rot90(target, rotations, dims=[1, 2])

        return {
            "lr_ms": lr_ms,
            "pan": pan,
            "target": target,
            "file": path.name,
        }


train_dataset = LocalWaldDataset(TRAIN_DIR, augment=True)
val_dataset = LocalWaldDataset(VAL_DIR, augment=False)
test_dataset = LocalWaldDataset(TEST_DIR, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(
    "Train/Val/Test:",
    len(train_dataset),
    len(val_dataset),
    len(test_dataset),
)

assert len(train_dataset) == 231
assert len(val_dataset) == 22
assert len(test_dataset) == 22


In [ ]:

class RefinementBlock(nn.Module):
    def __init__(self, channels=64):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, 1, 1),
        )

    def forward(self, tensor):
        return tensor + self.block(tensor)


class LocalHATPanFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.hat = HAT(
            upscale=4,
            in_chans=7,
            img_size=64,
            window_size=16,
            compress_ratio=3,
            squeeze_factor=30,
            conv_scale=0.01,
            overlap_ratio=0.5,
            img_range=1.0,
            depths=[6, 6, 6, 6, 6, 6],
            embed_dim=180,
            num_heads=[6, 6, 6, 6, 6, 6],
            mlp_ratio=2,
            upsampler="pixelshuffle",
            resi_connection="1conv",
            use_checkpoint=False,
            drop_path_rate=0.0,
        )

        self.hat.conv_last = nn.Conv2d(64, 6, 3, 1, 1)

        self.refine_head = nn.Conv2d(13, 64, 3, 1, 1)

        self.refine_body = nn.Sequential(
            RefinementBlock(64),
            RefinementBlock(64),
            RefinementBlock(64),
            RefinementBlock(64),
        )

        self.refine_tail = nn.Conv2d(64, 6, 3, 1, 1)

    def forward(self, lr_ms, pan_hr):
        bicubic = F.interpolate(
            lr_ms,
            size=pan_hr.shape[-2:],
            mode="bicubic",
            align_corners=False,
        )

        pan_lr = F.interpolate(
            pan_hr,
            size=lr_ms.shape[-2:],
            mode="area",
        )

        residual = self.hat(
            torch.cat(
                [lr_ms, pan_lr],
                dim=1,
            )
        )

        coarse = bicubic + residual

        refinement = self.refine_tail(
            self.refine_body(
                self.refine_head(
                    torch.cat(
                        [
                            coarse,
                            bicubic,
                            pan_hr,
                        ],
                        dim=1,
                    )
                )
            )
        )

        return (coarse + refinement).clamp(0, 1)


model = LocalHATPanFusion().to(device)
current_model = LocalHATPanFusion().to(device)

print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}",
)


## نقل أفضل EMA WV3 من 8 Bands إلى 6 Bands

In [ ]:

source_checkpoint = torch.load(
    LONG_WV3_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

source_state = source_checkpoint.get(
    "ema_state_dict",
    source_checkpoint["model_state_dict"],
)

target_state = model.state_dict()

adapted_keys = {
    "hat.conv_first.weight",
    "hat.conv_first.bias",
    "hat.conv_last.weight",
    "hat.conv_last.bias",
    "refine_head.weight",
    "refine_head.bias",
    "refine_tail.weight",
    "refine_tail.bias",
}

exact_transfers = 0

for key, value in source_state.items():
    if (
        key in target_state
        and key not in adapted_keys
        and target_state[key].shape == value.shape
    ):
        target_state[key] = value.clone()
        exact_transfers += 1

model.load_state_dict(target_state, strict=True)


def adapt_input(weight, target_channels):
    source_channels = weight.shape[1]
    mean_kernel = weight.mean(dim=1, keepdim=True)

    return mean_kernel.repeat(
        1,
        target_channels,
        1,
        1,
    ) * (
        source_channels / target_channels
    )


def adapt_output(weight, target_channels):
    mean_kernel = weight.mean(dim=0, keepdim=True)

    return mean_kernel.repeat(
        target_channels,
        1,
        1,
        1,
    )


def adapt_bias(bias, target_channels):
    return bias.mean().reshape(1).repeat(target_channels)


with torch.no_grad():
    model.hat.conv_first.weight.copy_(
        adapt_input(
            source_state["hat.conv_first.weight"],
            7,
        )
    )

    model.hat.conv_first.bias.copy_(
        source_state["hat.conv_first.bias"]
    )

    model.hat.conv_last.weight.copy_(
        adapt_output(
            source_state["hat.conv_last.weight"],
            6,
        )
    )

    model.hat.conv_last.bias.copy_(
        adapt_bias(
            source_state["hat.conv_last.bias"],
            6,
        )
    )

    source_refine = source_state["refine_head.weight"]

    source_coarse = source_refine[:, 0:8]
    source_bicubic = source_refine[:, 8:16]
    source_pan = source_refine[:, 16:17]

    target_refine = torch.cat(
        [
            adapt_input(source_coarse, 6),
            adapt_input(source_bicubic, 6),
            source_pan,
        ],
        dim=1,
    )

    model.refine_head.weight.copy_(target_refine)
    model.refine_head.bias.copy_(
        source_state["refine_head.bias"]
    )

    model.refine_tail.weight.copy_(
        adapt_output(
            source_state["refine_tail.weight"],
            6,
        )
    )

    model.refine_tail.bias.copy_(
        adapt_bias(
            source_state["refine_tail.bias"],
            6,
        )
    )

current_checkpoint = torch.load(
    CURRENT_LOCAL_HAT_PATH,
    map_location=device,
    weights_only=False,
)

current_model.load_state_dict(
    current_checkpoint["model_state_dict"],
    strict=True,
)

current_model.eval()

print("Exact tensors transferred:", exact_transfers)
print(
    "Source epoch:",
    source_checkpoint.get("epoch"),
)


In [ ]:

from copy import deepcopy


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.ema_model = deepcopy(model).eval()

        for parameter in self.ema_model.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        source_state = model.state_dict()
        ema_state = self.ema_model.state_dict()

        for key, ema_value in ema_state.items():
            source_value = source_state[key].detach()

            if ema_value.dtype.is_floating_point:
                ema_value.mul_(self.decay).add_(
                    source_value,
                    alpha=1.0 - self.decay,
                )
            else:
                ema_value.copy_(source_value)

    def state_dict(self):
        return self.ema_model.state_dict()

    def load_state_dict(self, state):
        self.ema_model.load_state_dict(state, strict=True)


ema = ModelEMA(model, decay=EMA_DECAY)


In [ ]:

from pytorch_msssim import ssim


def charbonnier_loss(prediction, target):
    difference = prediction - target

    return torch.sqrt(
        difference * difference
        + CHARBONNIER_EPS * CHARBONNIER_EPS
    ).mean()


def spectral_cosine_loss(prediction, target):
    prediction_dn = tensor_to_dn(prediction)
    target_dn = tensor_to_dn(target)

    dot = torch.sum(prediction_dn * target_dn, dim=1)

    denominator = torch.clamp(
        torch.linalg.vector_norm(prediction_dn, dim=1)
        * torch.linalg.vector_norm(target_dn, dim=1),
        min=1e-8,
    )

    cosine = torch.clamp(
        dot / denominator,
        -1,
        1,
    )

    return (1.0 - cosine).mean()


def total_loss(prediction, target, lr_ms):
    reconstruction = charbonnier_loss(prediction, target)

    structural = 1.0 - ssim(
        prediction,
        target,
        data_range=1.0,
        size_average=True,
    )

    spectral = spectral_cosine_loss(
        prediction,
        target,
    )

    downsampled = F.interpolate(
        prediction,
        size=lr_ms.shape[-2:],
        mode="area",
    )

    lr_consistency = F.l1_loss(
        downsampled,
        lr_ms,
    )

    total = (
        reconstruction
        + SSIM_WEIGHT * structural
        + SAM_WEIGHT * spectral
        + LR_CONSISTENCY_WEIGHT * lr_consistency
    )

    return total


def metric_values(prediction, target):
    mse = torch.mean(
        (prediction - target) ** 2,
        dim=(1, 2, 3),
    )

    psnr = 10.0 * torch.log10(
        1.0 / torch.clamp(mse, min=1e-12)
    )

    prediction_dn = tensor_to_dn(prediction)
    target_dn = tensor_to_dn(target)

    dot = torch.sum(
        prediction_dn * target_dn,
        dim=1,
    )

    denominator = torch.clamp(
        torch.linalg.vector_norm(prediction_dn, dim=1)
        * torch.linalg.vector_norm(target_dn, dim=1),
        min=1e-8,
    )

    sam = (
        torch.acos(
            torch.clamp(
                dot / denominator,
                -1 + 1e-7,
                1 - 1e-7,
            )
        )
        * 180.0
        / math.pi
    ).mean(dim=(1, 2))

    rmse = torch.sqrt(
        torch.mean(
            (prediction_dn - target_dn) ** 2,
            dim=(2, 3),
        )
    )

    target_mean = torch.mean(
        target_dn,
        dim=(2, 3),
    ).abs().clamp_min(1e-6)

    ergas = (
        100.0
        / 4.0
        * torch.sqrt(
            torch.mean(
                (rmse / target_mean) ** 2,
                dim=1,
            )
        )
    )

    return psnr, sam, ergas


In [ ]:

from tqdm.auto import tqdm


@torch.inference_mode()
def evaluate(evaluation_model, loader):
    evaluation_model.eval()

    values = {
        "l1": [],
        "psnr": [],
        "ssim": [],
        "sam": [],
        "ergas": [],
    }

    for batch in tqdm(
        loader,
        desc="Evaluation",
        leave=False,
    ):
        lr_ms = batch["lr_ms"].to(device)
        pan = batch["pan"].to(device)
        target = batch["target"].to(device)

        prediction = evaluation_model(
            lr_ms,
            pan,
        ).clamp(0, 1)

        psnr, sam, ergas = metric_values(
            prediction,
            target,
        )

        values["l1"].append(
            F.l1_loss(prediction, target).item()
        )

        values["psnr"].extend(
            psnr.cpu().tolist()
        )

        values["sam"].extend(
            sam.cpu().tolist()
        )

        values["ergas"].extend(
            ergas.cpu().tolist()
        )

        batch_ssim = ssim(
            prediction,
            target,
            data_range=1.0,
            size_average=False,
        )

        if batch_ssim.ndim == 0:
            values["ssim"].append(batch_ssim.item())
        else:
            values["ssim"].extend(
                batch_ssim.cpu().tolist()
            )

    return {
        key: float(np.mean(current_values))
        for key, current_values in values.items()
    }


## Stage A وStage B مع Resume

In [ ]:

from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau


def freeze_all():
    for parameter in model.parameters():
        parameter.requires_grad_(False)


def configure_stage_a():
    freeze_all()

    trainable_modules = [
        model.hat.conv_first,
        model.hat.conv_last,
        model.refine_head,
        model.refine_body,
        model.refine_tail,
    ]

    for module in trainable_modules:
        for parameter in module.parameters():
            parameter.requires_grad_(True)


def configure_stage_b():
    for parameter in model.parameters():
        parameter.requires_grad_(True)


def trainable_parameters():
    return [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]


def create_stage_optimizer(stage):
    learning_rate = (
        STAGE_A_LR
        if stage == "A"
        else STAGE_B_LR
    )

    optimizer = AdamW(
        trainable_parameters(),
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-7,
    )

    return optimizer, scheduler


In [ ]:

start_epoch = 1
best_psnr = -float("inf")
best_balanced_score = -float("inf")
history = []
patience_counter = 0

if LAST_PATH.exists():
    checkpoint = torch.load(
        LAST_PATH,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )

    ema.load_state_dict(
        checkpoint["ema_state_dict"]
    )

    start_epoch = int(checkpoint["epoch"]) + 1
    best_psnr = float(checkpoint.get("best_psnr", best_psnr))
    best_balanced_score = float(
        checkpoint.get(
            "best_balanced_score",
            best_balanced_score,
        )
    )
    history = checkpoint.get("history", [])
    patience_counter = int(
        checkpoint.get("patience_counter", 0)
    )

    resume_stage = checkpoint["stage"]

    if resume_stage == "A":
        configure_stage_a()
    else:
        configure_stage_b()

    optimizer, scheduler = create_stage_optimizer(
        resume_stage
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    print("Resume epoch:", start_epoch)
    print("Resume stage:", resume_stage)

else:
    configure_stage_a()
    optimizer, scheduler = create_stage_optimizer("A")

print("Training:", start_epoch, "→", TOTAL_EPOCHS)


In [ ]:

def save_checkpoint(
    path,
    epoch,
    stage,
    validation,
):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "ema_state_dict": ema.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "epoch": epoch,
            "stage": stage,
            "best_psnr": best_psnr,
            "best_balanced_score": best_balanced_score,
            "patience_counter": patience_counter,
            "validation": validation,
            "history": history,
        },
        path,
    )


for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    desired_stage = (
        "A"
        if epoch <= STAGE_A_EPOCHS
        else "B"
    )

    current_stage = (
        "A"
        if not model.hat.layers[0].blocks[0].attn.relative_position_bias_table.requires_grad
        else "B"
    )

    if desired_stage != current_stage:
        if desired_stage == "A":
            configure_stage_a()
        else:
            configure_stage_b()

        optimizer, scheduler = create_stage_optimizer(
            desired_stage
        )

        print(
            "Switched to Stage",
            desired_stage,
            "| Trainable:",
            f"{sum(p.numel() for p in trainable_parameters()):,}",
        )

    model.train()
    optimizer.zero_grad(set_to_none=True)

    losses = []

    progress = tqdm(
        train_loader,
        desc=f"Local HAT {epoch}/{TOTAL_EPOCHS} Stage {desired_stage}",
        leave=False,
    )

    for batch_index, batch in enumerate(progress, start=1):
        lr_ms = batch["lr_ms"].to(device)
        pan = batch["pan"].to(device)
        target = batch["target"].to(device)

        prediction = model(lr_ms, pan)

        if not torch.isfinite(prediction).all():
            raise FloatingPointError(
                f"Non-finite prediction at epoch {epoch}."
            )

        loss = total_loss(
            prediction,
            target,
            lr_ms,
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite loss at epoch {epoch}."
            )

        (loss / ACCUMULATION_STEPS).backward()

        should_step = (
            batch_index % ACCUMULATION_STEPS == 0
            or batch_index == len(train_loader)
        )

        if should_step:
            torch.nn.utils.clip_grad_norm_(
                trainable_parameters(),
                max_norm=GRADIENT_CLIP,
            )

            optimizer.step()
            ema.update(model)
            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item())
        progress.set_postfix(loss=f"{loss.item():.5f}")

    validation = evaluate(
        ema.ema_model,
        val_loader,
    )

    scheduler.step(validation["psnr"])

    balanced_score = (
        validation["psnr"]
        + 2.0 * validation["ssim"]
        - 0.08 * validation["sam"]
        - 0.05 * validation["ergas"]
    )

    history.append(
        {
            "epoch": epoch,
            "stage": desired_stage,
            "train_loss": float(np.mean(losses)),
            "validation": validation,
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
    )

    print(
        f"Epoch {epoch:03d} Stage {desired_stage}"
        f" | Train {np.mean(losses):.6f}"
        f" | PSNR {validation['psnr']:.3f}"
        f" | SSIM {validation['ssim']:.5f}"
        f" | SAM {validation['sam']:.3f}°"
        f" | ERGAS {validation['ergas']:.3f}"
        f" | L1 {validation['l1']:.5f}"
    )

    improved = False

    if validation["psnr"] > best_psnr:
        best_psnr = validation["psnr"]
        patience_counter = 0
        improved = True

        save_checkpoint(
            BEST_PSNR_PATH,
            epoch,
            desired_stage,
            validation,
        )

        print("Saved best PSNR.")

    if balanced_score > best_balanced_score:
        best_balanced_score = balanced_score

        save_checkpoint(
            BEST_BALANCED_PATH,
            epoch,
            desired_stage,
            validation,
        )

        print("Saved best balanced.")

    if not improved and desired_stage == "B":
        patience_counter += 1

    save_checkpoint(
        LAST_PATH,
        epoch,
        desired_stage,
        validation,
    )

    with open(HISTORY_PATH, "w", encoding="utf-8") as file:
        json.dump(
            history,
            file,
            indent=2,
            ensure_ascii=False,
        )

    if (
        desired_stage == "B"
        and patience_counter >= PATIENCE
    ):
        print("Early stopping.")
        break


## Test النهائي

In [ ]:

best_checkpoint = torch.load(
    BEST_PSNR_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    best_checkpoint["ema_state_dict"],
    strict=True,
)

model.eval()
current_model.eval()

method_names = [
    "Bicubic",
    "Current WV3-Transfer HAT",
    "Long-Pretrained HAT Transfer",
]

values = {
    name: {
        "psnr": [],
        "ssim": [],
        "sam": [],
        "ergas": [],
        "l1": [],
    }
    for name in method_names
}

with torch.inference_mode():
    for batch in tqdm(
        test_loader,
        desc="Final Test",
    ):
        lr_ms = batch["lr_ms"].to(device)
        pan = batch["pan"].to(device)
        target = batch["target"].to(device)

        bicubic = F.interpolate(
            lr_ms,
            size=target.shape[-2:],
            mode="bicubic",
            align_corners=False,
        ).clamp(0, 1)

        old_prediction = current_model(
            lr_ms,
            pan,
        ).clamp(0, 1)

        new_prediction = model(
            lr_ms,
            pan,
        ).clamp(0, 1)

        predictions = {
            "Bicubic": bicubic,
            "Current WV3-Transfer HAT": old_prediction,
            "Long-Pretrained HAT Transfer": new_prediction,
        }

        for name, prediction in predictions.items():
            psnr, sam, ergas = metric_values(
                prediction,
                target,
            )

            values[name]["psnr"].extend(psnr.cpu().tolist())
            values[name]["sam"].extend(sam.cpu().tolist())
            values[name]["ergas"].extend(ergas.cpu().tolist())
            values[name]["l1"].append(
                F.l1_loss(prediction, target).item()
            )

            batch_ssim = ssim(
                prediction,
                target,
                data_range=1.0,
                size_average=False,
            )

            if batch_ssim.ndim == 0:
                values[name]["ssim"].append(batch_ssim.item())
            else:
                values[name]["ssim"].extend(
                    batch_ssim.cpu().tolist()
                )

rows = []

for name in method_names:
    rows.append(
        {
            "Method": name,
            "PSNR": float(np.mean(values[name]["psnr"])),
            "SSIM": float(np.mean(values[name]["ssim"])),
            "SAM": float(np.mean(values[name]["sam"])),
            "ERGAS": float(np.mean(values[name]["ergas"])),
            "L1": float(np.mean(values[name]["l1"])),
        }
    )

table = pd.DataFrame(rows)

table.to_csv(
    TEST_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

payload = {
    "checkpoint_epoch": int(best_checkpoint["epoch"]),
    "selection": "best EMA validation PSNR",
    "metrics": rows,
}

with open(TEST_JSON_PATH, "w", encoding="utf-8") as file:
    json.dump(
        payload,
        file,
        indent=2,
        ensure_ascii=False,
    )

display(table.round(6))

print("Saved:", TEST_CSV_PATH)
print("Saved:", TEST_JSON_PATH)



# التشغيل

ابدأ بـ:

```python
RUN_PROFILE = "smoke"
```

ثم:

```python
RUN_PROFILE = "long"
```

ولأقصى تدريب:

```python
RUN_PROFILE = "max"
```

الـResume تلقائي من:

```text
WV3_Long_Transfer_HAT_6Band/last_long_transfer_hat_6band.pth
```

لا تعتمد النموذج الجديد قبل ظهور جدول Test النهائي ومقارنته بالنتيجة الحالية:

```text
Current HAT PSNR ≈ 25.357
Current HAT SSIM ≈ 0.9210
Current HAT SAM ≈ 1.9107°
Current HAT ERGAS ≈ 2.1864
```
